In [28]:
import numpy as np
import pandas as pd
import re
import pickle
import json
import time
import os
import tensorflow as tf
from tensorflow.keras.layers import Embedding, LSTM, Dense, Layer, Input
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [29]:
tf.random.set_seed(42)
np.random.seed(42)

In [30]:
df = pd.read_csv("Dataset_English_Hindi.csv")
df.drop_duplicates(inplace=True)
df = df.dropna()
df["English"] = df["English"].astype(str)
df["Hindi"] = df["Hindi"].astype(str)
print(f"Total pairs: {len(df)}")

Total pairs: 127375


In [31]:
def clean_english(sentence):
    sentence = sentence.lower().strip()
    sentence = re.sub(r'[^a-zA-Z0-9\s]', '', sentence)
    sentence = re.sub(r'https?://\S+|www\.\S+', '', sentence)
    emoji_pattern = re.compile("["
        u"\U0001F600-\U0001F64F"
        u"\U0001F300-\U0001F5FF"
        u"\U0001F680-\U0001F6FF"
        u"\U0001F1E0-\U0001F1FF"
        u"\U00002702-\U000027B0"
        u"\U000024C2-\U0001F251"
        "]+", flags=re.UNICODE)
    sentence = emoji_pattern.sub(r'', sentence)
    return sentence.strip()

In [32]:
def clean_hindi(sentence):
    sentence = sentence.strip()
    sentence = re.sub(r'[^\u0900-\u097F\s]', '', sentence)
    sentence = re.sub(r'https?://\S+|www\.\S+', '', sentence)
    return sentence.strip()

In [33]:
df["English"] = df["English"].apply(clean_english)
df["Hindi"] = df["Hindi"].apply(clean_hindi)

In [34]:
MAX_ENG_WORDS = 25
MAX_HIN_WORDS = 30
mask = (
    df["English"].str.split().str.len().between(5, MAX_ENG_WORDS) &
    df["Hindi"].str.split().str.len().between(5, MAX_HIN_WORDS)
)
df = df[mask].reset_index(drop=True)
print(f"Pairs after length filter: {len(df)}")

Pairs after length filter: 88576


In [35]:
# Add <start> / <end> tokens to Hindi
df["Hindi_in"] = "<sos> " + df["Hindi"]   # decoder input
df["Hindi_out"] = df["Hindi"] + " <eos>"  # decoder target

In [36]:
# English tokenizer
eng_tok = Tokenizer(filters="", lower=True, oov_token="<OOV>")
eng_tok.fit_on_texts(df["English"])
ENG_VOCAB = len(eng_tok.word_index) + 1

In [37]:
# Hindi tokenizer (case-sensitive for Devanagari)
hin_tok = Tokenizer(filters="", lower=False, oov_token="<OOV>")
hin_tok.fit_on_texts(df["Hindi_in"].tolist() + df["Hindi_out"].tolist())
HIN_VOCAB = len(hin_tok.word_index) + 1

In [38]:
print(f"English vocab: {ENG_VOCAB} | Hindi vocab: {HIN_VOCAB}")

English vocab: 52611 | Hindi vocab: 54090


In [39]:
df_enc_inp = pad_sequences(
    eng_tok.texts_to_sequences(df["English"]),
    maxlen=MAX_ENG_WORDS, padding="post"
)
df_dec_inp = pad_sequences(
    hin_tok.texts_to_sequences(df["Hindi_in"]),
    maxlen=MAX_HIN_WORDS + 1, padding="post"
)
df_dec_out = pad_sequences(
    hin_tok.texts_to_sequences(df["Hindi_out"]),
    maxlen=MAX_HIN_WORDS + 1, padding="post"
)

In [40]:
ENG_SEQ_LEN = df_enc_inp.shape[1]
HIN_SEQ_LEN = df_dec_inp.shape[1]
print(f"Encoder seq len: {ENG_SEQ_LEN} | Decoder seq len: {HIN_SEQ_LEN}")
print(f"df_enc_inp shape: {df_enc_inp.shape}")
print(f"df_dec_out shape: {df_dec_out.shape}")

Encoder seq len: 25 | Decoder seq len: 31
df_enc_inp shape: (88576, 25)
df_dec_out shape: (88576, 31)


In [41]:
EMBED_DIM = 512
BATCH_SIZE = 32
EPOCHS = 25

In [42]:
# Custom Luong Attention Layer
class LuongAttention(Layer):
    """
    Luong 'general' attention:
        score(s_t, h_i) = s_t^T * W_a * h_i
        alpha            = softmax(scores)
        context          = sum(alpha * encoder_outputs)
    """
    def __init__(self, units, **kwargs):
        super(LuongAttention, self).__init__(**kwargs)
        self.units = units
        self.W = Dense(units, use_bias=False, name="attention_W")

    def call(self, decoder_outputs, encoder_outputs):
        # decoder_outputs : (batch, dec_len, units)
        # encoder_outputs : (batch, enc_len, units)

        # Linear transform on decoder side → (batch, dec_len, units)
        query = self.W(decoder_outputs)

        # Dot with encoder outputs → scores (batch, dec_len, enc_len)
        scores = tf.matmul(query, encoder_outputs, transpose_b=True)
        attention_weights = tf.nn.softmax(scores, axis=-1)

        # Weighted sum of encoder outputs → (batch, dec_len, units)
        context = tf.matmul(attention_weights, encoder_outputs)

        return context, attention_weights

    def get_config(self):
        config = super().get_config()
        config.update({"units": self.units})
        return config

In [43]:
# Encoder
# CHANGED: return_sequences=True  →  gives all timestep outputs for attention
enc_input = Input(shape=(ENG_SEQ_LEN,))
enc_embed = Embedding(ENG_VOCAB, EMBED_DIM)(enc_input)
enc_lstm  = LSTM(EMBED_DIM, return_sequences=True, return_state=True, name="enc_lstm")
enc_all_out, enc_h, enc_c = enc_lstm(enc_embed)

enc_model = Model(enc_input, [enc_all_out, enc_h, enc_c])

In [44]:
# Decoder
dec_input = Input(shape=(HIN_SEQ_LEN,))
dec_embed = Embedding(HIN_VOCAB, EMBED_DIM, name="dec_embedding")(dec_input)
dec_lstm  = LSTM(EMBED_DIM, return_sequences=True, return_state=True, name="dec_lstm")
dec_out, dec_h, dec_c = dec_lstm(dec_embed, initial_state=[enc_h, enc_c])

In [45]:
# Luong Attention
attention_layer = LuongAttention(EMBED_DIM, name="luong_attention")
context, _ = attention_layer(dec_out, enc_all_out)

In [46]:
#concat_out = tf.concat([dec_out, context], axis=-1)        # (batch, dec_len, 2*EMBED_DIM)
concat_out = tf.keras.layers.Concatenate(axis=-1, name="concat_attn_train")([dec_out, context])
attn_dense = Dense(EMBED_DIM, activation='tanh', name="attn_dense")
attn_out   = attn_dense(concat_out)                        # (batch, dec_len, EMBED_DIM)

dec_dense    = Dense(HIN_VOCAB, activation='softmax', name="dec_dense")
dec_final_out = dec_dense(attn_out)

In [47]:
#Model
model = Model([enc_input, dec_input], dec_final_out)
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

Model: "model_5"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_6 (InputLayer)           [(None, 25)]         0           []                               
                                                                                                  
 input_7 (InputLayer)           [(None, 31)]         0           []                               
                                                                                                  
 embedding_1 (Embedding)        (None, 25, 512)      26936832    ['input_6[0][0]']                
                                                                                                  
 dec_embedding (Embedding)      (None, 31, 512)      27694080    ['input_7[0][0]']                
                                                                                            

In [48]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
early_stopping = EarlyStopping(monitor='accuracy', patience=5, restore_best_weights=True)

In [49]:
# Training
history = model.fit(
    x=[df_enc_inp, df_dec_inp],
    y=df_dec_out,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    callbacks=[early_stopping]
)

Epoch 1/25
2768/2768 [==============================] - 201s 72ms/step - loss: 3.4241 - accuracy: 0.5805
Epoch 2/25
2768/2768 [==============================] - 200s 72ms/step - loss: 3.1465 - accuracy: 0.6021
Epoch 3/25
2768/2768 [==============================] - 201s 73ms/step - loss: 2.9632 - accuracy: 0.6153
Epoch 4/25
2768/2768 [==============================] - 202s 73ms/step - loss: 2.6499 - accuracy: 0.6355
Epoch 5/25
2768/2768 [==============================] - 202s 73ms/step - loss: 2.1321 - accuracy: 0.6667
Epoch 6/25
2768/2768 [==============================] - 201s 73ms/step - loss: 1.6451 - accuracy: 0.7111
Epoch 7/25
2768/2768 [==============================] - 201s 73ms/step - loss: 1.2629 - accuracy: 0.7539
Epoch 8/25
2768/2768 [==============================] - 201s 73ms/step - loss: 0.9746 - accuracy: 0.7971
Epoch 9/25
2768/2768 [==============================] - 202s 73ms/step - loss: 0.7740 - accuracy: 0.8310
Epoch 10/25
2768/2768 [==============================] 

In [50]:
model.save("my_model.keras")
model.save_weights("my_model.weights.h5")
with open("my_model_architecture.json", 'w') as f:
    f.write(model.to_json())

history_df = pd.DataFrame(history.history)
history_df.index = range(1, len(history_df) + 1)
history_df.index.name = "epoch"
history_df.to_csv("training_history.csv")

In [51]:
# Inference
# Encoder inference: returns all outputs + final states
encoder_inf_model = Model(enc_input, [enc_all_out, enc_h, enc_c])

# Reuse trained layers
dec_emb_layer   = model.get_layer("dec_embedding")
dec_dense_layer = model.get_layer("dec_dense")
attn_dense_layer = model.get_layer("attn_dense")

# Decoder inference inputs
dec_inf_input      = Input(shape=(1,))
dec_state_inp_h    = Input(shape=(EMBED_DIM,))
dec_state_inp_c    = Input(shape=(EMBED_DIM,))
enc_outputs_input  = Input(shape=(ENG_SEQ_LEN, EMBED_DIM), name="enc_outputs_input")  # NEW

dec_inf_embed = dec_emb_layer(dec_inf_input)
dec_inf_out, dec_inf_h, dec_inf_c = dec_lstm(
    dec_inf_embed, initial_state=[dec_state_inp_h, dec_state_inp_c]
)

# Single-step attention
context_inf, _ = attention_layer(dec_inf_out, enc_outputs_input)
concat_inf = tf.keras.layers.Concatenate(axis=-1, name="concat_attn_inf")([dec_inf_out, context_inf])
attn_inf_out    = attn_dense_layer(concat_inf)
dec_inf_dense   = dec_dense_layer(attn_inf_out)

decoder_inf_model = Model(
    inputs  = [dec_inf_input, dec_state_inp_h, dec_state_inp_c, enc_outputs_input],
    outputs = [dec_inf_dense, dec_inf_h, dec_inf_c]
)

In [52]:
# Translate
hin_idx2word = {idx: word for word, idx in hin_tok.word_index.items()}
sos_token    = hin_tok.word_index["<sos>"]
eos_token    = hin_tok.word_index["<eos>"]

In [53]:
def translate(english_sentence, max_len=MAX_HIN_WORDS):
    # Step 1: Clean and pad English input
    eng_clean = clean_english(english_sentence)
    eng_seq   = eng_tok.texts_to_sequences([eng_clean])
    eng_pad   = pad_sequences(eng_seq, maxlen=ENG_SEQ_LEN, padding="post")

    # Step 2: Encode → get all outputs + initial states
    enc_all_outputs, h, c = encoder_inf_model.predict(eng_pad, verbose=0)

    # Step 3: Start decoding with <sos>
    target_token = np.zeros((1, 1))
    target_token[0, 0] = sos_token

    translated = []

    # Step 4: Decode one token at a time
    for _ in range(max_len):
        pred, h, c = decoder_inf_model.predict(
            [target_token, h, c, enc_all_outputs], verbose=0
        )

        predicted_id   = np.argmax(pred[0, 0, :])
        predicted_word = hin_idx2word.get(predicted_id, "")

        if predicted_id == eos_token or predicted_word == "":
            break

        translated.append(predicted_word)
        target_token[0, 0] = predicted_id  # feed output as next input

    return " ".join(translated)

In [54]:
# Test
test_sentences = [
    "How are you doing today",
    "I want to eat food",
    "The weather is very cold outside",
    "She is going to the market",
    "My name is Rahul",
    "I love my country",
    "He is a very good student",
    "Please open the door",
    "I am feeling very tired today",
    "The dog is running in the park",
    "Can you help me please",
    "I want to drink water",
    "She loves to read books",
    "The sun rises in the east",
    "I go to school every day",
    "He does not like cold weather",
    "We are going to the temple",
    "The train is very late today",
    "I want to sleep now",
    "My mother cooks very delicious food"
]

for s in test_sentences:
    print(f"English : {s}")
    print(f"Hindi   : {translate(s)}")
    print("-" * 80)

English : How are you doing today
Hindi   : आज आप यह कर रहे हैं
--------------------------------------------------------------------------------
English : I want to eat food
Hindi   : मैं खाना खाने देना चाहता हूँ।
--------------------------------------------------------------------------------
English : The weather is very cold outside
Hindi   : घर की गर्मी बहुत ही ठंडा हो रही है।
--------------------------------------------------------------------------------
English : She is going to the market
Hindi   : वे बाज़ार कबूल करना चाहते हैं ।
--------------------------------------------------------------------------------
English : My name is Rahul
Hindi   : मेरा नाम अहमद है।
--------------------------------------------------------------------------------
English : I love my country
Hindi   : मुझे देश की पसंद करता है।
--------------------------------------------------------------------------------
English : He is a very good student
Hindi   : उसको एक विद्यार्थी अच्छी तरह से अच्छा बर्ताव होत